# Bootstrap dos dados

Este notebook automatiza o download dos dados usados nos demais notebooks deste repositório, a partir do dataset [`rikdifos/credit-card-approval-prediction`](https://www.kaggle.com/datasets/rikdifos/credit-card-approval-prediction) no Kaggle, usando a biblioteca [`kagglehub`](https://github.com/Kaggle/kagglehub).

Rode este notebook uma única vez, antes de `demo_criacao_target.ipynb`, para popular `../data/raw/` com `application_record.csv` e `credit_record.csv`. Se você preferir baixar os arquivos manualmente, veja a seção "Obtendo os dados" do `README.md` na raiz do repositório — este notebook é só uma alternativa automatizada, não obrigatória.

## Autenticação no Kaggle

O `kagglehub` precisa de uma credencial de API do Kaggle para baixar o dataset. Configure uma das opções abaixo antes de rodar a célula de download:

- **Arquivo de credenciais**: baixe `kaggle.json` em [kaggle.com/settings](https://www.kaggle.com/settings) (seção "API" → "Create New Token") e salve em `~/.kaggle/kaggle.json`.
- **Variáveis de ambiente**: defina `KAGGLE_USERNAME` e `KAGGLE_KEY` no shell antes de iniciar o Jupyter.
- **Login interativo**: rode `kagglehub.login()` numa célula — ele solicita as credenciais e as guarda para as próximas execuções.

Sem uma dessas três opções configuradas, a célula de download abaixo falha com um erro de autenticação.

In [1]:
from pathlib import Path

DATA_DIR = Path("../data/raw")
DATASET = "rikdifos/credit-card-approval-prediction"

APPLICATION_PATH = DATA_DIR / "application_record.csv"
CREDIT_PATH = DATA_DIR / "credit_record.csv"


## Download e cópia dos arquivos

A célula abaixo é idempotente: se os dois CSVs já existirem em `../data/raw/`, ela não baixa nada de novo.

In [2]:
import shutil

import kagglehub

if APPLICATION_PATH.exists() and CREDIT_PATH.exists():
    print(f"Os dois arquivos já estão presentes em {DATA_DIR.resolve()} — nada a fazer.")
else:
    DATA_DIR.mkdir(parents=True, exist_ok=True)

    cache_path = Path(kagglehub.dataset_download(DATASET))
    print(f"Dataset baixado em: {cache_path}")

    csv_files = list(cache_path.glob("*.csv"))
    if not csv_files:
        raise FileNotFoundError(f"Nenhum CSV encontrado em {cache_path}")

    for csv_file in csv_files:
        destino = DATA_DIR / csv_file.name
        shutil.copy(csv_file, destino)
        print(f"Copiado: {csv_file.name} -> {destino}")


Os dois arquivos já estão presentes em /home/drl/fiap/dtat/2dtat-bb/data/raw — nada a fazer.


/home/drl/fiap/dtat/2dtat-bb/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Verificação final

Confere se os arquivos esperados existem e dá uma espiada nas primeiras linhas de cada um.

In [3]:
import pandas as pd
from IPython.display import display

for path in (APPLICATION_PATH, CREDIT_PATH):
    assert path.exists(), f"Arquivo esperado não encontrado: {path}"
    tamanho_mb = path.stat().st_size / 1e6
    print(f"{path.name}: OK ({tamanho_mb:.1f} MB)")
    display(pd.read_csv(path, nrows=2))


application_record.csv: OK (54.3 MB)


,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS
0,5008804,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2.0
1,5008805,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2.0


credit_record.csv: OK (15.4 MB)


,ID,MONTHS_BALANCE,STATUS
0,5001711,0,X
1,5001711,-1,0


## Próximos passos

Com os CSVs em `../data/raw/`, siga para [`demo_criacao_target.ipynb`](demo_criacao_target.ipynb).